# Deploy VSS Warehouse Blueprint 3.2

This notebook deploys the NVIDIA VSS Warehouse Blueprint 3.2 on GPU-equipped cloud instances.

In 3.2 the warehouse compose files ship **in-tree** in the `video-search-and-summarization` repo — there is no separate NGC compose tarball. On Brev, the repo is cloned onto the instance automatically; set `DEPLOY_SOURCE_PATH` in Section 1 to that path (typically `~/video-search-and-summarization`).

**What it does:**
1. Validates GPU hardware and system prerequisites (Phase 2)
2. Installs and configures the NGC CLI (Phase 1)
3. Configures Docker storage for large image pulls
4. Downloads the warehouse app data from NGC (Phase 4 — compose files come from the repo)
5. Configures `industry-profiles/warehouse-operations/.env` and deploys with Docker Compose (Phases 5 & 8)
6. Monitors and verifies the deployment (Phase 9)
7. Provides Agent UI & Kibana access links


## 1. Configuration

Set your NGC API key and warehouse deployment parameters below.
All subsequent cells use these variables.

**Required to set yourself:**

- `NGC_CLI_API_KEY` — get from https://ngc.nvidia.com
- `DEPLOY_SOURCE_PATH` — path where the `video-search-and-summarization` repo is cloned. Default is `~/video-search-and-summarization`. The warehouse compose files now live in-tree at `<DEPLOY_SOURCE_PATH>/deploy/docker/`.

**Profile combinations:**

| MODE | BP_PROFILE | STREAM_TYPE | Dataset | Streams |
|------|-----------|-------------|---------|---------|
| `2d` | `bp_wh_kafka` | `kafka` | `warehouse-loading-dock-3cams-synthetic` | 3 |
| `2d` | `bp_wh_redis` | `redis` | `warehouse-loading-dock-3cams-synthetic` | 3 |
| `2d` | `bp_wh` | `kafka` | `nv-warehouse-4cams` | 4 |
| `3d` | `bp_wh_kafka` | `kafka` | `warehouse-4cams-20mx20m-synthetic` | 4 |
| `3d` | `bp_wh_redis` | `redis` | `warehouse-4cams-20mx20m-synthetic` | 4 |
| `mv3dt` | `bp_wh_kafka` | `kafka` | `warehouse-4cams-20mx20m-synthetic` | 4 |
| `mv3dt` | `bp_wh_redis` | `redis` | `warehouse-4cams-20mx20m-synthetic` | 4 |

> `bp_wh` (agents) is **2D-only**. `mv3dt` and `3d` support kafka/redis only. `STREAM_TYPE` is always `kafka` for `bp_wh`.


In [ ]:
# ============================================================
# REQUIRED: Set these before running anything else
# ============================================================

NGC_CLI_API_KEY = ""          # Your NGC API key — get one at https://ngc.nvidia.com
NGC_CLI_ORG = ""              # Optional NGC org. Leave empty to infer from APP_DATA_RESOURCE.

# ---- Repo path (Brev clones the repo automatically; override if it's elsewhere) ----
import os as _os
DEPLOY_SOURCE_PATH = _os.path.expanduser("~/video-search-and-summarization")

# ---- Deployment mode ----
MODE = "2d"                   # "2d" | "3d" | "mv3dt"

# ---- Blueprint profile ----
# Valid combinations (see blueprint-deploy.sh):
#   MODE=2d:    bp_wh (default), bp_wh_kafka, bp_wh_redis
#   MODE=3d:    bp_wh_kafka, bp_wh_redis  (bp_wh not valid)
#   MODE=mv3dt: bp_wh_kafka, bp_wh_redis  (bp_wh not valid)
BP_PROFILE = "bp_wh_kafka"    # "bp_wh_kafka" | "bp_wh_redis" | "bp_wh"
STREAM_TYPE = "kafka"         # "kafka" | "redis"  (auto-synced with BP_PROFILE; defaults to kafka for bp_wh, which does not use a broker)

# ---- Hardware profile ----
# Valid: H100, L40, L40S, L4, A6000, RTXA6000, RTXA6000ADA, RTXPRO6000BW, IGX-THOR, DGX-SPARK
HARDWARE_PROFILE = "RTXPRO6000BW"

# ---- Minimal vs extended (bp_wh_kafka / bp_wh_redis only, any mode) ----
# "true"  = minimal  — excludes ELK, Video Analytics API/UI, Monitoring
# ""      = extended — full deployment (includes Kibana on port 5601)
# Ignored for bp_wh (it has a fixed service set).
MINIMAL_PROFILE = ""

# ---- LLM (bp_wh only) ----
LLM_MODE = "none"             # "none" | "local" | "remote"
# Remote LLM (LLM_MODE="remote") — point at an external OpenAI-compatible endpoint.
# Required when LLM_MODE="remote"; ignored for local/none.
# LLM options:
# - LLM_NAME: nvidia/nvidia-nemotron-nano-9b-v2
#   LLM_NAME_SLUG: nvidia-nemotron-nano-9b-v2
# - LLM_NAME: nvidia/nemotron-3-nano
#   LLM_NAME_SLUG: nemotron-3-nano
# - LLM_NAME: nvidia/llama-3.3-nemotron-super-49b-v1.5
#   LLM_NAME_SLUG: llama-3.3-nemotron-super-49b-v1.5
# - LLM_NAME: openai/gpt-oss-20b
#   LLM_NAME_SLUG: gpt-oss-20b
LLM_NAME="nvidia/nvidia-nemotron-nano-9b-v2"
LLM_NAME_SLUG="none"
LLM_BASE_URL = "https://integrate.api.nvidia.com/v1"             # e.g. "https://integrate.api.nvidia.com/v1" (build.nvidia.com)
NVIDIA_API_KEY = ""           # required for build.nvidia.com remote endpoints
OPENAI_API_KEY = ""           # required instead for OpenAI remote endpoints

# ---- App-data NGC resource ----
# Default published resource path:
#   nvidia/vss-warehouse/vss-warehouse-app-data:3.2.0
# Uses the published nvidia/vss-warehouse resource path.
APP_DATA_RESOURCE = "nvidia/vss-warehouse/vss-warehouse-app-data:3.2.0"

# ---- Dataset and stream count ----
# Set both explicitly. Recommended pairings by mode + profile:
#   MODE=2d  + BP_PROFILE=bp_wh                                  → SAMPLE_VIDEO_DATASET="nv-warehouse-4cams"                     | NUM_STREAMS=4
#   MODE=2d  + BP_PROFILE=bp_wh_kafka / bp_wh_redis              → SAMPLE_VIDEO_DATASET="warehouse-loading-dock-3cams-synthetic" | NUM_STREAMS=3
#   MODE=3d  + BP_PROFILE=bp_wh_kafka / bp_wh_redis → SAMPLE_VIDEO_DATASET="warehouse-4cams-20mx20m-synthetic" | NUM_STREAMS=4
#   MODE=mv3dt + BP_PROFILE=bp_wh_kafka / bp_wh_redis → SAMPLE_VIDEO_DATASET="warehouse-4cams-20mx20m-synthetic" | NUM_STREAMS=4
SAMPLE_VIDEO_DATASET = "warehouse-loading-dock-3cams-synthetic"
NUM_STREAMS = 3

# ============================================================
# OPTIONAL: Override defaults if needed
# ============================================================

# Download directory for NGC artifacts (leave empty to use home dir)
DOWNLOAD_DIR = ""             # e.g. "/home/ubuntu"

# Network overrides (auto-detected in Section 7 if left empty)
HOST_IP_OVERRIDE = ""         # Internal IP — leave empty for auto-detect
EXTERNAL_IP_OVERRIDE = ""     # External IP — leave empty for auto-detect (browser-facing IP on NAT'd clouds)


In [ ]:
# ---- Validate and resolve configuration ----
import os

assert NGC_CLI_API_KEY, "NGC_CLI_API_KEY is required. Get one at https://ngc.nvidia.com"
assert MODE in ("2d", "3d", "mv3dt"), f"Invalid MODE: {MODE!r}. Must be '2d', '3d', or 'mv3dt'."
assert BP_PROFILE in ("bp_wh_kafka", "bp_wh_redis", "bp_wh"), (
    f"Invalid BP_PROFILE: {BP_PROFILE!r}. "
    "Must be 'bp_wh_kafka', 'bp_wh_redis', or 'bp_wh'."
)
if MODE in ("3d", "mv3dt") and BP_PROFILE == "bp_wh":
    raise ValueError(
        f"MODE={MODE} is not compatible with BP_PROFILE=bp_wh. "
        "MODE=3d and MODE=mv3dt support bp_wh_kafka / bp_wh_redis only "
        "(bp_wh requires MODE=2d because Agents only run in 2D)."
    )

assert SAMPLE_VIDEO_DATASET, (
    "SAMPLE_VIDEO_DATASET is required. See Section 1 for the recommended pairings by MODE + BP_PROFILE."
)
assert isinstance(NUM_STREAMS, int) and NUM_STREAMS > 0, (
    f"NUM_STREAMS must be a positive integer (got {NUM_STREAMS!r}). "
    "See Section 1 for the recommended pairings by MODE + BP_PROFILE."
)

# Resolve and verify the repo path — compose lives in-tree under <repo>/deploy/docker
DEPLOY_SOURCE_PATH = os.path.expanduser(DEPLOY_SOURCE_PATH)
assert os.path.isdir(DEPLOY_SOURCE_PATH), (
    f"DEPLOY_SOURCE_PATH does not exist: {DEPLOY_SOURCE_PATH!r}. "
    "On Brev the repo is cloned automatically — set DEPLOY_SOURCE_PATH in Section 1."
)
DEPLOYMENTS_DIR = os.path.join(DEPLOY_SOURCE_PATH, "deploy", "docker")
WAREHOUSE_ENV   = os.path.join(DEPLOYMENTS_DIR, "industry-profiles", "warehouse-operations", ".env")
assert os.path.isfile(os.path.join(DEPLOYMENTS_DIR, "compose.yml")), (
    f"compose.yml not found at {DEPLOYMENTS_DIR}/."
)
assert os.path.isfile(WAREHOUSE_ENV), f"Warehouse .env not found at {WAREHOUSE_ENV}."

# Sync STREAM_TYPE with BP_PROFILE — kafka for bp_wh (no broker)
# matches deploy/docker/scripts/blueprint-deploy.sh:821-826.
if "kafka" in BP_PROFILE:
    STREAM_TYPE = "kafka"
elif "redis" in BP_PROFILE:
    STREAM_TYPE = "redis"
else:
    STREAM_TYPE = "kafka"

# LLM is only meaningful for MODE=2d + BP_PROFILE=bp_wh (NIM + agents stack);
# every other combo forces LLM_MODE="none" (blueprint-deploy.sh:754-762).
if BP_PROFILE == "bp_wh":
    if LLM_MODE == "none": LLM_MODE = "local"
else:
    LLM_MODE = "none"
# Remote LLM requires an external endpoint URL + an API key (no local NIM is deployed).
if LLM_MODE == "remote":
    assert LLM_BASE_URL, (
        "LLM_MODE='remote' requires LLM_BASE_URL (e.g. https://integrate.api.nvidia.com/v1). "
        "Set it in Section 1."
    )
    assert NVIDIA_API_KEY or OPENAI_API_KEY, (
        "LLM_MODE='remote' requires NVIDIA_API_KEY (build.nvidia.com) or OPENAI_API_KEY (OpenAI). "
        "Set one in Section 1."
    )

# LLM_NAME_SLUG names the local LLM NIM container and is only meaningful for
# LLM_MODE=local. For remote/none no NIM is deployed (only llm_local_<slug>
# compose profiles exist), so force "none" — matches blueprint-deploy.sh.
if LLM_MODE != "local":
    LLM_NAME_SLUG = "none"

# Warehouse ALWAYS uses the integrated RTVI VLM (vss-rtvi-vlm), never the
# standalone VLM NIM path — there is no vlm_* slice in COMPOSE_PROFILES. So
# VLM_MODE is "none" for every warehouse profile, matching the warehouse .env
# (VLM_MODE=none / VLM_NAME_SLUG=none). blueprint-deploy.sh keeps the .env's
# "none" for bp_wh rather than promoting it to "local".
VLM_MODE = "none"

# Export NGC key to environment for shell cells
os.environ["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY

# Resolve app-data extract path (NGC extracts to vss-warehouse-app-data_v<version>/vss-warehouse-app-data/)
_base = DOWNLOAD_DIR.rstrip("/") if DOWNLOAD_DIR else os.path.expanduser("~")
_app_data_version = APP_DATA_RESOURCE.split(":", 1)[-1]
APP_DATA_DIR = os.path.join(_base, f"vss-warehouse-app-data_v{_app_data_version}", "vss-warehouse-app-data")

print("Configuration:")
print(f"  DEPLOY_SOURCE_PATH: {DEPLOY_SOURCE_PATH}")
print(f"  DEPLOYMENTS_DIR:    {DEPLOYMENTS_DIR}")
print(f"  WAREHOUSE_ENV:      {WAREHOUSE_ENV}")
print(f"  MODE:               {MODE}")
print(f"  BP_PROFILE:         {BP_PROFILE}")
print(f"  STREAM_TYPE:        {STREAM_TYPE}")
print(f"  HARDWARE_PROFILE:   {HARDWARE_PROFILE}")
print(f"  MINIMAL_PROFILE:    {MINIMAL_PROFILE!r}  ({'minimal' if MINIMAL_PROFILE == 'true' else 'extended'})")
print(f"  DATASET:            {SAMPLE_VIDEO_DATASET}")
print(f"  NUM_STREAMS:        {NUM_STREAMS}")
print(f"  LLM_MODE:           {LLM_MODE}")
if LLM_MODE != "none":
    print(f"  LLM_NAME:           {LLM_NAME}  (slug: {LLM_NAME_SLUG})")
if LLM_MODE == "remote":
    print(f"  LLM_BASE_URL:       {LLM_BASE_URL}")
print(f"  APP_DATA_RESOURCE:  {APP_DATA_RESOURCE}")
print(f"  APP_DATA_DIR:       {APP_DATA_DIR}")
print(f"  NGC org:            {NGC_CLI_ORG or '(inferred from APP_DATA_RESOURCE)'}")
print(f"  NGC key:            {NGC_CLI_API_KEY[:4]}...{NGC_CLI_API_KEY[-4:]}")
print()
print("Configuration valid.")


## 2. Prerequisites Check

Verifies all system requirements for the Warehouse Blueprint (Phase 2):

- **2.1** NVIDIA driver and GPU detection
- **2.2** Docker 28.3.3+ (tested range [28.3.3, 29.5.0); see Section 4.1), non-root access, cgroupfs driver
- **2.3** NVIDIA Container Toolkit
- **2.4** Linux kernel network settings (IPv6 disabled, socket buffers)
- **2.5** IPv6 localhost entry in `/etc/hosts`
- **2.6** Minimum resources: 10+ CPU cores, 64 GB+ RAM, 500 GB+ disk

If a check fails, install/fix the missing component before proceeding.


In [ ]:
%%bash

echo "=== 2.1 GPU & NVIDIA Driver ==="
nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader
echo ""

echo "=== 2.2 Docker ==="
docker --version
docker compose version
if docker ps > /dev/null 2>&1; then
    echo "Docker non-root access: OK"
else
    echo "WARNING: Docker requires sudo."
    echo "  Fix: sudo usermod -aG docker $USER && newgrp docker"
fi
echo ""

echo "=== cgroupfs driver ==="
if docker info 2>/dev/null | grep -qi "cgroupfs"; then
    echo "cgroupfs: OK"
else
    echo "WARNING: cgroupfs driver not configured in /etc/docker/daemon.json"
    echo "  Fix: add exec-opts native.cgroupdriver=cgroupfs, then restart docker"
fi
echo ""

echo "=== 2.3 NVIDIA Container Toolkit ==="
if docker run --rm --gpus all ubuntu:22.04 nvidia-smi > /dev/null 2>&1; then
    echo "NVIDIA Container Toolkit: OK"
else
    echo "WARNING: NVIDIA Container Toolkit not working."
    echo "  Install: https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html"
fi
echo ""

echo "=== 2.4 Linux Kernel Settings ==="
IPV6=$(sysctl -n net.ipv6.conf.all.disable_ipv6 2>/dev/null || echo "unset")
RMEM=$(sysctl -n net.core.rmem_max 2>/dev/null || echo "0")
echo "net.ipv6.conf.all.disable_ipv6 = $IPV6 (need: 1)"
echo "net.core.rmem_max = $RMEM (need: 5242880)"
if [ "$IPV6" != "1" ]; then
    echo "WARNING: IPv6 not disabled."
fi
if [ "${RMEM:-0}" -lt 5242880 ] 2>/dev/null; then
    echo "WARNING: rmem_max too low."
fi
echo ""

echo "=== 2.5 IPv6 Localhost Entry ==="
HOSTS_ENTRY=$(grep "^::1" /etc/hosts 2>/dev/null || echo "(not found)")
echo "  /etc/hosts ::1 line: $HOSTS_ENTRY"
if echo "$HOSTS_ENTRY" | grep -q "localhost6"; then
    echo "  IPv6 localhost entry: OK"
elif echo "$HOSTS_ENTRY" | grep -q "^::1"; then
    echo "  WARNING: must use localhost6 not localhost"
    echo "    Fix: sudo sed -i 's/^::1 localhost ip6/::1 localhost6 ip6/' /etc/hosts"
else
    echo "  INFO: No ::1 entry found (OK if IPv6 is disabled)"
fi
echo ""

echo "=== 2.6 Minimum System Resources ==="
echo "CPU cores: $(nproc) (need: 10+)"
free -h | awk '/^Mem:/ {print "RAM: " $2 " (need: 64 GB+)"}'
df -h / | tail -1 | awk '{print "Disk free: " $4 " of " $2 " (need: 500 GB+)"}'
echo ""

echo "Prerequisites check complete."


## 3. Install NGC CLI

The NGC CLI is required to download the warehouse app-data. This cell installs it if not already present, then configures it with your API key.


In [ ]:
import subprocess, os, shutil

def run(cmd, **kwargs):
    """Run a shell command, raise on failure with output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()

# Check if NGC CLI is already installed
ngc_path = shutil.which("ngc")
if ngc_path:
    ver = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {ver}")
else:
    import platform
    arch = platform.machine()
    if arch in ("aarch64", "arm64"):
        filename = "ngccli_linux_arm64.zip"
    else:
        filename = "ngccli_linux.zip"

    NGC_CLI_VERSION = "4.13.0"
    url = f"https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/versions/{NGC_CLI_VERSION}/files/{filename}"

    print(f"Installing NGC CLI {NGC_CLI_VERSION} ...")
    run(f"cd /tmp && wget -q --content-disposition '{url}' -O ngc_cli.zip")
    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(f"NGC CLI download failed — file is only {size} bytes. Check the version URL.")
    print(f"  Downloaded {size / 1024 / 1024:.1f} MB")

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    ver = run("ngc --version 2>&1 | head -1")
    print(f"  Installed: {ver}")

# Configure NGC CLI with API key. Prefer explicit NGC_CLI_ORG from Section 1;
# fall back to inferring from APP_DATA_RESOURCE's first path segment for back-compat.
_ngc_org = NGC_CLI_ORG or APP_DATA_RESOURCE.split("/", 1)[0]
print(f"Configuring NGC CLI (org={_ngc_org})...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)
with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = {_ngc_org}
""")

print("NGC CLI configured.")
print(run("ngc config current"))


## 4. Docker & Containerd Storage

Docker images and containerd layers for VSS require **~250GB** (NIM models, DeepStream, ELK, etc.). Most GPU cloud instances ship with a small root disk (200-250GB) that **will run out of space** during deployment.

This cell auto-detects whether your root disk is too small and moves Docker and containerd storage to a larger mount. Docker **volumes** (Elasticsearch indices, uploaded videos, Kafka data) are kept on the root disk so your data persists even if the instance is stopped and the ephemeral NVMe is wiped. Images and layers are re-pulled automatically on next deploy.

**Common NVMe mount points** (auto-detected):
- AWS DLAMI: `/opt/dlami/nvme`
- Brev/Crusoe: `/ephemeral`
- Custom RAID: `/data`

To override auto-detection, set `STORAGE_ROOT` below.

In [ ]:
import subprocess, json, os, shutil

STORAGE_ROOT = ""  # Override: set to a mount path (e.g. "/mnt/data") to skip auto-detection

MIN_ROOT_FREE_GB = 350  # If root has less than this free, move storage

# --- Auto-detect large mount ---

def get_disk_free_gb(path):
    """Return free space in GB for the filesystem containing path."""
    st = os.statvfs(path)
    return (st.f_bavail * st.f_frsize) / (1024 ** 3)

def get_disk_total_gb(path):
    st = os.statvfs(path)
    return (st.f_blocks * st.f_frsize) / (1024 ** 3)

def find_large_mount():
    """Look for a large non-root mount suitable for Docker storage."""
    candidates = ["/opt/dlami/nvme", "/ephemeral", "/data"]
    for path in candidates:
        if os.path.isdir(path) and os.path.ismount(path):
            free = get_disk_free_gb(path)
            if free > 200:
                return path, free
    return None, 0

def find_mount_unit(mount_path):
    """Convert a mount path to a systemd mount unit name (e.g. /opt/dlami/nvme -> opt-dlami-nvme.mount)."""
    # Strip leading slash, replace remaining slashes with dashes
    unit = mount_path.strip("/").replace("/", "-") + ".mount"
    # Verify this unit exists on the system
    r = subprocess.run(["systemctl", "cat", unit], capture_output=True, text=True)
    if r.returncode == 0:
        return unit
    return None

root_free = get_disk_free_gb("/")
root_total = get_disk_total_gb("/")

print(f"Root disk: {root_free:.0f} GB free / {root_total:.0f} GB total")

if STORAGE_ROOT:
    large_mount = STORAGE_ROOT
    mount_free = get_disk_free_gb(STORAGE_ROOT)
    print(f"Using override: {STORAGE_ROOT} ({mount_free:.0f} GB free)")
    need_move = True
else:
    large_mount, mount_free = find_large_mount()
    need_move = root_free < MIN_ROOT_FREE_GB and large_mount is not None

    if large_mount:
        print(f"Large mount:    {large_mount} ({mount_free:.0f} GB free)")
    else:
        print("No large ephemeral mount detected.")

    if root_free >= MIN_ROOT_FREE_GB:
        print(f"\nRoot disk has enough space ({root_free:.0f} GB free). No storage move needed.")
    elif not large_mount:
        print(f"\nWARNING: Root disk only has {root_free:.0f} GB free and no large mount was found.")
        print("Deployment may fail due to disk space. Consider attaching a larger volume.")

if need_move:
    DOCKER_DATA_ROOT = os.path.join(large_mount, "docker")
    CONTAINERD_ROOT = os.path.join(large_mount, "containerd")
    VOLUMES_DIR = "/var/lib/docker/volumes"  # Keep volumes on persistent root disk

    print(f"\nMoving Docker and containerd storage to {large_mount}")
    print(f"  Docker images/layers: {DOCKER_DATA_ROOT}")
    print(f"  Containerd:           {CONTAINERD_ROOT}")
    print(f"  Docker volumes:       {VOLUMES_DIR} (stays on root for persistence)")

    # --- Check what needs changing ---
    daemon_json = "/etc/docker/daemon.json"
    config = {}
    try:
        with open(daemon_json) as f:
            config = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        pass

    need_daemon_json = config.get("data-root") != DOCKER_DATA_ROOT

    subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT], check=True)
    subprocess.run(["sudo", "mkdir", "-p", VOLUMES_DIR], check=True)

    volumes_link = os.path.join(DOCKER_DATA_ROOT, "volumes")
    need_volumes_symlink = not (os.path.islink(volumes_link) and os.readlink(volumes_link) == VOLUMES_DIR)

    containerd_link = "/var/lib/containerd"
    need_containerd = not (os.path.islink(containerd_link) and os.readlink(containerd_link) == CONTAINERD_ROOT)

    # Even if symlinks are correct, ensure NVMe target dirs actually exist
    # (they get wiped when ephemeral NVMe is reset on instance stop/start)
    need_target_dirs = not os.path.isdir(DOCKER_DATA_ROOT) or not os.path.isdir(CONTAINERD_ROOT)
    if need_target_dirs:
        print(f"\n  NVMe target dir(s) missing — recreating...")
        subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT, CONTAINERD_ROOT], check=True)

    if not need_daemon_json and not need_volumes_symlink and not need_containerd:
        print(f"\n  Docker data-root already set to {DOCKER_DATA_ROOT}")
        print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")
        print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

        # Always ensure the boot-time restore service is up to date
        # (handles the case where service exists but is missing mount dependencies)
        _update_restore_service = True
        _need_restart = need_target_dirs  # Restart Docker/containerd if we had to recreate dirs
    else:
        _update_restore_service = True
        _need_restart = True

        # Stop Docker AND docker.socket (socket can reactivate Docker and recreate dirs)
        print("\n  Stopping Docker and containerd for storage reconfiguration...")
        subprocess.run(["sudo", "systemctl", "stop", "docker.socket"], check=False)
        subprocess.run(["sudo", "systemctl", "stop", "docker"], check=True)
        subprocess.run(["sudo", "systemctl", "stop", "containerd"], check=True)

        # --- Docker daemon.json ---
        if need_daemon_json:
            config["data-root"] = DOCKER_DATA_ROOT
            new_config = json.dumps(config, indent=2)
            subprocess.run(
                f"echo '{new_config}' | sudo tee {daemon_json}",
                shell=True, check=True, capture_output=True
            )
            print(f"  Docker data-root set to {DOCKER_DATA_ROOT}")
        else:
            print(f"  Docker data-root already set to {DOCKER_DATA_ROOT}")

        # --- Volumes symlink (use ln -sfn for idempotency) ---
        if need_volumes_symlink:
            # ln -sfn: force, no-dereference (replaces existing dir/symlink atomically)
            subprocess.run(["sudo", "rm", "-rf", volumes_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", VOLUMES_DIR, volumes_link], check=True)
            print(f"  Created symlink: {volumes_link} -> {VOLUMES_DIR}")
        else:
            print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")

        # --- Containerd ---
        if need_containerd:
            subprocess.run(["sudo", "mkdir", "-p", CONTAINERD_ROOT], check=True)
            if os.path.isdir(containerd_link) and not os.path.islink(containerd_link):
                # Move existing containerd data
                subprocess.run(f"sudo mv {containerd_link}/* {CONTAINERD_ROOT}/ 2>/dev/null; true",
                               shell=True, check=False)
                subprocess.run(["sudo", "rm", "-rf", containerd_link], check=True)
                print(f"  Containerd data moved to {CONTAINERD_ROOT}")
            elif os.path.lexists(containerd_link):
                subprocess.run(["sudo", "rm", "-f", containerd_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", CONTAINERD_ROOT, containerd_link], check=True)
            print(f"  Containerd symlinked: {containerd_link} -> {CONTAINERD_ROOT}")
        else:
            print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

    # --- Install/update boot-time restore service ---
    # Ephemeral NVMe is wiped on instance stop/start. This systemd service
    # recreates the directories before Docker/containerd start so they don't crash-loop.
    # We use RequiresMountsFor= so the service waits for the NVMe to actually be mounted.
    if _update_restore_service:
        unit_name = "docker-nvme-restore.service"
        unit_path = f"/etc/systemd/system/{unit_name}"

        # Build After= line — include the mount unit if systemd knows about it
        after_targets = "local-fs.target"
        mount_unit = find_mount_unit(large_mount)
        if mount_unit:
            after_targets += f" {mount_unit}"

        unit_content = f"""[Unit]
Description=Restore Docker/containerd dirs on ephemeral NVMe
Before=containerd.service docker.service
After={after_targets}
RequiresMountsFor={large_mount}

[Service]
Type=oneshot
ExecStart=/bin/bash -c 'mkdir -p {DOCKER_DATA_ROOT} {CONTAINERD_ROOT}'

[Install]
WantedBy=multi-user.target
"""
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.service', delete=False) as tmp:
            tmp.write(unit_content)
            tmp_path = tmp.name
        subprocess.run(["sudo", "cp", tmp_path, unit_path], check=True)
        os.unlink(tmp_path)
        subprocess.run(["sudo", "systemctl", "daemon-reload"], check=True)
        subprocess.run(["sudo", "systemctl", "enable", unit_name], check=True, capture_output=True)
        print(f"  Installed {unit_name} (restores NVMe dirs on boot, waits for mount)")

    # --- Restart if needed ---
    if _need_restart:
        print("\n  Starting containerd and Docker...")
        subprocess.run(["sudo", "systemctl", "start", "containerd"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker.socket"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker"], check=True)

    r = subprocess.run(["docker", "info", "--format", "{{.DockerRootDir}}"],
                       capture_output=True, text=True)
    print(f"\n  Docker data-root: {r.stdout.strip()}")
    target = os.readlink(containerd_link) if os.path.islink(containerd_link) else containerd_link
    print(f"  Containerd root:  {target}")
    print(f"\n  Storage configuration complete.")
else:
    if not STORAGE_ROOT and root_free >= MIN_ROOT_FREE_GB:
        print("Skipping storage move.")

### 4.1 Pin Docker version

Pin Docker CE + plugins + containerd.io to a known-good combination (CE **29.4.3**, buildx **0.33.0**, compose **5.1.3**, containerd **2.2.3**). Some Brev launchables ship newer versions than the Warehouse blueprint is tested against; pin explicitly so compose/buildx incompatibilities don't surface in later sections. `apt-mark hold` prevents unattended-upgrades or later cells from drifting the box back.

The cell first reads the installed Docker Engine version: if it already falls in the tested range **[28.3.3, 29.5.0)** the version downgrade is **skipped** — re-pinning to an exact version the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64) would fail with *version not found* for no benefit. The packages are still `apt-mark hold`-ed at their current versions so the box can't drift past the tested range mid-run. Safe to re-run.

Runs *after* the section 4 storage relocation so the APT download lands on the relocated volume and the dockerd restart triggered by the downgrade picks up the new data-root.

In [ ]:
%%bash
# Pin Docker CE + plugins + containerd.io to a known-good combination, but
# ONLY when the host's Docker is outside the tested range. Some Brev
# launchables ship a newer Docker than the Warehouse blueprint is tested
# against; pin explicitly so compose/buildx incompatibilities don't surface
# mid-deployment.
#
# When the installed Docker already falls in [28.3.3, 29.5.0) the version
# downgrade is skipped: re-pinning to an exact epoch-versioned package that
# the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64)
# fails with "version not found" for no benefit. The in-range packages are
# still held so the box can't drift past the tested range mid-notebook.
# Idempotent -- safe to re-run.

set -euo pipefail

# Tested Docker Engine range -- keep in sync with the Warehouse prereq check
# and warehouse.md (28.3.3 / Compose v2.39.1+).
MIN_DOCKER_VERSION="28.3.3"
MAX_DOCKER_VERSION="29.5.0"

# Packages frozen with `apt-mark hold` so unattended-upgrades / later
# `apt-get install` calls can't drift the box before the notebook finishes.
HOLD_PKGS="docker-ce docker-ce-cli docker-buildx-plugin docker-compose-plugin containerd.io"

version_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]; }
version_lt() { [ "$1" != "$2" ] && [ "$(printf '%s\n%s\n' "$1" "$2" | sort -V | head -n1)" = "$1" ]; }

DOCKER_VERSION="$(docker version --format '{{.Server.Version}}' 2>/dev/null || true)"
if [ -n "$DOCKER_VERSION" ] \
   && version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
   && version_lt "$DOCKER_VERSION" "$MAX_DOCKER_VERSION"; then
  echo "Docker $DOCKER_VERSION is within the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); skipping the Docker version pin."
  # No downgrade needed, but still hold the in-range packages at their
  # current versions so unattended-upgrades / later apt-get calls can't drift
  # the box past the tested range for the remainder of the notebook.
  sudo apt-mark hold $HOLD_PKGS
  exit 0
fi

if [ -n "$DOCKER_VERSION" ]; then
  echo "Docker $DOCKER_VERSION is outside the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); pinning to known-good versions."
else
  echo "Could not read the installed Docker version; pinning to known-good versions."
fi

# Read distro info from /etc/os-release (always present on Ubuntu; minimal
# images don't ship `lsb_release`).
. /etc/os-release
DISTRO="${VERSION_ID}"
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME}}"

# Versions hard-coded to what shipped alongside docker-ce 29.4.3 on the
# Docker apt repo (verified against download.docker.com + upstream GitHub
# release timestamps). When bumping DOCKER_CE_VER, bump these four together.
DOCKER_CE_VER="5:29.4.3-1~ubuntu.${DISTRO}~${CODENAME}"
BUILDX_VER="0.33.0-1~ubuntu.${DISTRO}~${CODENAME}"
COMPOSE_VER="5.1.3-1~ubuntu.${DISTRO}~${CODENAME}"
CONTAINERD_VER="2.2.3-1~ubuntu.${DISTRO}~${CODENAME}"

# Refresh the APT cache first -- without this, the specific epoch-versioned
# package may not be in the local index and the install would fail with
# version-not-found before any pinning takes effect.
sudo apt-get update -qq

sudo DEBIAN_FRONTEND=noninteractive apt-get install -y \
  --allow-downgrades \
  -o Dpkg::Options::=--force-confdef \
  -o Dpkg::Options::=--force-confold \
  docker-ce="$DOCKER_CE_VER" \
  docker-ce-cli="$DOCKER_CE_VER" \
  docker-buildx-plugin="$BUILDX_VER" \
  docker-compose-plugin="$COMPOSE_VER" \
  containerd.io="$CONTAINERD_VER"

# Hold so unattended-upgrades / later `apt-get install` calls don't drift
# the box back to newer versions before the rest of the notebook runs.
sudo apt-mark hold $HOLD_PKGS

## 5. Docker Login

Authenticate with the NVIDIA Container Registry (`nvcr.io`) to pull deployment images.

In [ ]:
import subprocess

result = subprocess.run(
    ["docker", "login", "nvcr.io",
     "--username", "$oauthtoken",
     "--password", NGC_CLI_API_KEY],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Docker login to nvcr.io: OK")
else:
    print(f"Docker login FAILED:\n{result.stderr}")
    raise RuntimeError("Docker login to nvcr.io failed")

## 6. Get Deployment Code

**Compose bundle (3.2 change):** the warehouse compose files ship **in-tree** in the `video-search-and-summarization` repo — there is no separate compose tarball to download. On Brev, the repo is cloned automatically; Section 1 verified `DEPLOY_SOURCE_PATH` points at it.

**App data** (videos + pre-trained models) still comes from NGC:

- Default: `nvidia/vss-warehouse/vss-warehouse-app-data:3.2.0`
- `APP_DATA_RESOURCE` defaults to the published `nvidia/vss-warehouse/vss-warehouse-app-data:3.2.0` resource.

> **First run only** — if the app-data directory already exists on disk, the download is skipped.


In [ ]:
import subprocess, os

_base = DOWNLOAD_DIR.rstrip("/") if DOWNLOAD_DIR else os.path.expanduser("~")

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()

# ---- Compose bundle: in-tree, no download ----
print(f"Compose bundle (in-tree): {DEPLOYMENTS_DIR}")
print(f"  compose.yml:    {os.path.join(DEPLOYMENTS_DIR, 'compose.yml')}")
print(f"  warehouse .env: {WAREHOUSE_ENV}")
print()

# ---- App data ----
_app_data_version = APP_DATA_RESOURCE.split(":", 1)[-1]
APP_DATA_EXTRACT_DIR = os.path.join(_base, f"vss-warehouse-app-data_v{_app_data_version}")
APP_DATA_DIR         = os.path.join(APP_DATA_EXTRACT_DIR, "vss-warehouse-app-data")

if os.path.isdir(APP_DATA_DIR):
    print(f"App data already present: {APP_DATA_DIR}")
else:
    print(f"Downloading app data ({APP_DATA_RESOURCE}) ...")
    run(f"ngc registry resource download-version {APP_DATA_RESOURCE!r}", cwd=_base)
    tarball = os.path.join(APP_DATA_EXTRACT_DIR, "vss-warehouse-app-data.tar.gz")
    print(f"Extracting {os.path.basename(tarball)} ...")
    run(f"tar -xf {tarball!r}", cwd=APP_DATA_EXTRACT_DIR)
    print("App data extracted.")

run(f"sudo chmod -R 777 {APP_DATA_DIR!r}")
print(f"APP_DATA_DIR:    {APP_DATA_DIR}")
print()
print("Deployment artifacts ready.")


## 7. Detect Network Configuration

Auto-detects internal (`HOST_IP`) and external (`EXTERNAL_IP`) addresses. On NAT'd cloud instances (Brev, AWS), these are different — the internal IP is used for inter-container communication while the external IP is used for browser access.

If auto-detection fails or gives the wrong result, set the overrides in Section 1.


In [ ]:
import subprocess, os

def detect_internal_ip():
    """Detect internal IP via `ip route` (same method as dev-profile.sh)."""
    try:
        out = subprocess.run(
            ["bash", "-c", "ip route get 1.1.1.1 | awk '/src/ {for (i=1;i<=NF;i++) if ($i==\"src\") print $(i+1)}'"],
            capture_output=True, text=True, timeout=5,
        )
        return out.stdout.strip()
    except Exception:
        return ""

def detect_external_ip():
    """Detect external IP via a public echo service."""
    for cmd in ["curl -s --max-time 5 ifconfig.me", "curl -s --max-time 5 icanhazip.com"]:
        try:
            out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
            ip = out.stdout.strip()
            if ip:
                return ip
        except Exception:
            continue
    return ""

def read_etc_environment():
    """Read key=value pairs from /etc/environment (Brev sets BREV_ENV_ID there)."""
    env = {}
    try:
        with open("/etc/environment") as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    key, _, value = line.partition("=")
                    env[key.strip()] = value.strip().strip('"')
    except FileNotFoundError:
        pass
    return env

HOST_IP = HOST_IP_OVERRIDE or detect_internal_ip()
EXTERNAL_IP = EXTERNAL_IP_OVERRIDE or detect_external_ip()

print(f"Internal IP (HOST_IP):   {HOST_IP}")
print(f"External IP:             {EXTERNAL_IP}")

if HOST_IP and EXTERNAL_IP and HOST_IP == EXTERNAL_IP:
    print("\nInternal == External (direct connection, no NAT)")
elif HOST_IP and EXTERNAL_IP:
    print("\nNAT detected — internal and external IPs differ.")
    print("Browser-facing URLs will use EXTERNAL_IP.")

if not HOST_IP:
    print("\nWARNING: Could not detect internal IP. Set HOST_IP_OVERRIDE in Section 1.")
if not EXTERNAL_IP:
    print("\nWARNING: Could not detect external IP. Set EXTERNAL_IP_OVERRIDE in Section 1.")

# --- Brev Secure Links ---
# On Brev, all browser-facing traffic routes through the HAProxy ingress
# on a single port (default 7777). This avoids CORS issues with Cloudflare
# Access when each port gets its own hostname.
# Check os.environ first, then fall back to /etc/environment (Jupyter kernels
# may not inherit /etc/environment depending on how the notebook server starts).
_etc_env = read_etc_environment()
BREV_ENV_ID = os.environ.get("BREV_ENV_ID") or _etc_env.get("BREV_ENV_ID", "")
if BREV_ENV_ID:
    # Ensure it's in os.environ so downstream tooling picks it up
    os.environ["BREV_ENV_ID"] = BREV_ENV_ID
    proxy_port = os.environ.get("PROXY_PORT", "7777")
    # Brev secure links prefix the haproxy port directly (e.g. port 7777
    # → "7777-xxx.brevlab.com"). Older launchables used to add a trailing
    # "0" giving "77770-..."; that form is legacy. Set BREV_LINK_PREFIX
    # to override if you inherit an instance still using it.
    brev_link_prefix = os.environ.get("BREV_LINK_PREFIX", f"{proxy_port}")
    os.environ["BREV_LINK_PREFIX"] = brev_link_prefix
    brev_ui_url = f"https://{brev_link_prefix}-{BREV_ENV_ID}.brevlab.com"
    print(f"\n=== Brev Environment Detected ===")
    print(f"  BREV_ENV_ID:         {BREV_ENV_ID}")
    print(f"  Secure link prefix:  {brev_link_prefix} (set BREV_LINK_PREFIX to override)")
    print(f"  All browser-facing URLs route through HAProxy ingress (port {proxy_port})")
    print(f"  UI will be available at: {brev_ui_url}")
else:
    BREV_ENV_ID = ""  # ensure defined for later cells
    print("\nNo Brev environment detected")


## 8. Deploy

This cell (Phases 5 & 8):
1. Writes the warehouse `.env` at `<repo>/deploy/docker/industry-profiles/warehouse-operations/.env` with your settings from Section 1 (template defaults are preserved for everything else)
2. Logs in to `nvcr.io`
3. Runs `docker compose up` from `<repo>/deploy/docker/`:

   ```bash
   docker compose -f compose.yml \
     --env-file industry-profiles/warehouse-operations/.env \
     up --detach --pull always --force-recreate --build
   ```

**Expected time: 10–20 minutes** on first run (image pulls + perception engine build).
Subsequent runs are faster — images stay in the local cache and the TensorRT engine is cached under `$VSS_DATA_DIR/models/`.

Full log is written to `~/deploy_warehouse.log`. Monitor progress in Section 9.


In [ ]:
import os, re, shlex, subprocess, time

LOG_FILE = os.path.expanduser("~/deploy_warehouse.log")
ENV_FILE = WAREHOUSE_ENV  # <repo>/deploy/docker/industry-profiles/warehouse-operations/.env

def merge_warehouse_dotenv(path, assignments):
    """Update KEY=value lines in the warehouse .env in place; preserve all other variables.

    Returns the set of keys written by this merge (replaced or appended).
    """
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    used = set()
    out = []
    for line in lines:
        s = line.strip()
        if not s or s.startswith("#"):
            out.append(line)
            continue
        if "=" not in s:
            out.append(line)
            continue
        key, _, _ = s.partition("=")
        key = key.strip()
        if key in assignments:
            val = assignments[key]
            out.append(f"{key}={shlex.quote(val)}\n")
            used.add(key)
        else:
            out.append(line)
    for k in assignments:
        if k not in used:
            out.append(f"{k}={shlex.quote(assignments[k])}\n")
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(out)
    return set(assignments)

# ---- Phase 5: Merge into <repo>/deploy/docker/industry-profiles/warehouse-operations/.env ----
print(f"Merging notebook settings into {ENV_FILE} ...")
assert os.path.isfile(ENV_FILE), f"Missing {ENV_FILE}. Re-run Section 6."

_env_updates = {
    "MODE": MODE,
    "BP_PROFILE": BP_PROFILE,
    "STREAM_TYPE": STREAM_TYPE,
    "MINIMAL_PROFILE": MINIMAL_PROFILE,
    "SAMPLE_VIDEO_DATASET": SAMPLE_VIDEO_DATASET,
    "NUM_STREAMS": str(NUM_STREAMS),
    "HARDWARE_PROFILE": HARDWARE_PROFILE,
    "LLM_MODE": LLM_MODE,
    "LLM_NAME": LLM_NAME,
    "LLM_NAME_SLUG": LLM_NAME_SLUG,
    "VLM_MODE": VLM_MODE,
    "VSS_APPS_DIR": DEPLOYMENTS_DIR,
    "VSS_DATA_DIR": APP_DATA_DIR,
    "HOST_IP": HOST_IP,
    "EXTERNAL_IP": EXTERNAL_IP or HOST_IP,
    "NGC_CLI_API_KEY": NGC_CLI_API_KEY,
}

# Remote LLM (bp_wh + LLM_MODE=remote): write the external endpoint + credentials.
# Only written when set, so local/none deployments keep the .env defaults.
if LLM_MODE == "remote" and LLM_BASE_URL:
    _env_updates["LLM_BASE_URL"] = LLM_BASE_URL
if NVIDIA_API_KEY:
    _env_updates["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if OPENAI_API_KEY:
    _env_updates["OPENAI_API_KEY"] = OPENAI_API_KEY

# Brev secure-link overrides — when running behind Brev's edge proxy, the
# inbound Host header is "<port>-<env>.brevlab.com", not the instance IP.
# HAProxy's h_main ACL (services/infra/haproxy/haproxy.cfg.template:97-106)
# only matches VSS_PUBLIC_HOST / EXTERNAL_IP / HOST_IP / localhost, so without
# the override every path-based route (/kibana/, /vst/, ...) returns 404.
# Mirrors dev-profile.sh:1143-1152 which does the same for the VSS notebook.
if BREV_ENV_ID:
    _proxy_port = os.environ.get("PROXY_PORT", "7777")
    _brev_host = f"{_proxy_port}-{BREV_ENV_ID}.brevlab.com"
    _env_updates["HAPROXY_PORT"] = _proxy_port
    _env_updates["VSS_PUBLIC_HTTP_PROTOCOL"] = "https"
    _env_updates["VSS_PUBLIC_WS_PROTOCOL"] = "wss"
    _env_updates["VSS_PUBLIC_HOST"] = _brev_host
    _env_updates["VSS_PUBLIC_PORT"] = "443"
    # Only the VSS_PUBLIC_* vars (+ HAPROXY_PORT) are overridden for Brev.
    print(f"Brev override applied: VSS_PUBLIC_HOST={_brev_host}, VSS_PUBLIC_PORT=443 ")
_written = merge_warehouse_dotenv(ENV_FILE, _env_updates)
_preview = {k: (f"{v[:4]}...{v[-4:]}" if k in ("NGC_CLI_API_KEY", "NVIDIA_API_KEY", "OPENAI_API_KEY") and len(v) > 12 else v) for k, v in _env_updates.items()}
print(f"warehouse .env updated (template defaults preserved). Keys merged: {sorted(_written)}")
print(f"Values (NGC_CLI_API_KEY redacted): {_preview}")
print()

# ---- Phase 8: Bring up ----
print("Logging in to nvcr.io ...")
r = subprocess.run(
    ["docker", "login", "--username", "$oauthtoken", "--password", NGC_CLI_API_KEY, "nvcr.io"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise RuntimeError(f"Docker login failed:\n{r.stderr}")
print("Docker login: OK")
print()

# Run docker compose from <repo>/deploy/docker.
_cmd = [
    "docker", "compose",
    "--progress", "plain",
    "-f", "compose.yml",
    "--env-file", "industry-profiles/warehouse-operations/.env",
    "up",
    "--detach",
    "--pull", "always",
    "--force-recreate",
    "--build",
]
_compose_env = {**os.environ, "NGC_CLI_API_KEY": NGC_CLI_API_KEY, "NO_COLOR": "1"}

# --- Parse docker compose plain output ---
PULLING_RE = re.compile(r"(?:^|\s)(\S+)\s+Pulling\b|Pulling\s+(?:from\s+)?(\S+)")
PULLED_RE = re.compile(r"(?:✔\s*)?(\S+)\s+Pulled\b")
BUILD_STEP_RE = re.compile(r"#\d+\s+\[([^\]]+)\]\s+(.+)")
IMAGE_BUILT_RE = re.compile(r"(?:✔\s*)?(\S+)\s+Built\b|Successfully\s+built\s+(\S+)")
CONTAINER_RE = re.compile(r"^\s*Container\s+(\S+)\s+(.+?)\s*$")

SUPPRESS_PATTERNS = [re.compile(r"^\s*$"), re.compile(r"^#[0-9]+\s+\[internal\]")]
PHASE_PATTERNS = [
    (re.compile(r"\[\+\]\s*Pulling", re.I), "pulling"),
    (re.compile(r"\[\+\]\s*Building", re.I), "building"),
    (re.compile(r"Creating network|Network\s+\S+\s+Created", re.I), "network"),
    (re.compile(r"Creating\s|Created\s+container", re.I), "containers"),
]

errors = []
images_pulling = set()
images_pulled = set()
builds = {}
builds_done = set()
containers = {}
phases_seen = []
STATUS_REFRESH_INTERVAL_S = 10
last_refresh = 0.0
_t0 = time.monotonic()


def _elapsed():
    return time.monotonic() - _t0


def print_status():
    print("=" * 50, flush=True)
    print(f"Elapsed {_elapsed():.0f}s  |  errors: {len(errors)}", flush=True)
    print(
        f"  pull: {len(images_pulling)} in flight, {len(images_pulled)} done  |  "
        f"build services: {len(builds)} active, {len(builds_done)} done  |  "
        f"containers: {len(containers)}",
        flush=True,
    )
    if phases_seen:
        print("  phases:", " → ".join(phases_seen[-8:]), flush=True)
    if builds:
        tail = list(builds.items())[-3:]
        print("  build:", "; ".join(f"{k}: {v[:48]}" for k, v in tail), flush=True)
    if containers:
        tail = list(containers.items())[-5:]
        print("  last containers:", "; ".join(f"{n}={s}" for n, s in tail), flush=True)
    if errors:
        print(f"  last error line: {errors[-1][:200]}", flush=True)
    print("=" * 50, flush=True)


print(f"Command: {' '.join(_cmd)}")
print(f"Working dir: {DEPLOYMENTS_DIR}")
print(f"Logging to {LOG_FILE} — status keeps updating when something changes.")
print()

process = subprocess.Popen(
    _cmd,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=DEPLOYMENTS_DIR,
    env=_compose_env,
)
assert process.stdout is not None

rc = None
try:
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for line in process.stdout:
            log.write(line)
            log.flush()
            stripped = line.rstrip()

            if "[ERROR]" in stripped:
                errors.append(stripped)
                print_status()
                continue

            m_pulling = PULLING_RE.search(stripped)
            if m_pulling:
                name = next((g for g in m_pulling.groups() if g), None)
                if name:
                    images_pulling.add(name)
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()
                continue

            m_pulled = PULLED_RE.search(stripped)
            if m_pulled:
                name = m_pulled.group(1)
                images_pulled.add(name)
                images_pulling.discard(name)
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()
                continue

            m_build = BUILD_STEP_RE.match(stripped)
            if m_build:
                svc, step = m_build.group(1), m_build.group(2)
                builds[svc] = step
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()
                continue

            m_built = IMAGE_BUILT_RE.search(stripped)
            if m_built:
                svc = next((g for g in m_built.groups() if g), None)
                if svc:
                    builds_done.add(svc)
                    builds.pop(svc, None)
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()
                continue

            if any(p.search(stripped) for p in SUPPRESS_PATTERNS):
                continue

            for pattern, label in PHASE_PATTERNS:
                if pattern.search(stripped):
                    if label not in phases_seen:
                        phases_seen.append(label)
                        print_status()
                    break

            m = CONTAINER_RE.match(stripped)
            if m:
                name, status = m.group(1), m.group(2)
                if status.startswith("Exited"):
                    status = "Exited"
                containers[name] = status
                now = time.monotonic()
                if now - last_refresh > STATUS_REFRESH_INTERVAL_S:
                    last_refresh = now
                    print_status()

        rc = process.wait()
        log.write(f"\n---- docker compose finished (exit code {rc}) ----\n")
        log.flush()

    print_status()
    print("=" * 50)
    if rc == 0 and not errors:
        print(f"Deployment complete in {_elapsed():.1f}s.")
    else:
        print(f"\nDeployment FAILED (exit code {rc}).")
        if errors:
            print(f"\n{len(errors)} error line(s) captured — see above and log.")
        print(f"\nFull log: {LOG_FILE}")
        print(f"  View with: cat {LOG_FILE}")

except KeyboardInterrupt:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=120)
        except Exception:
            pass
    raise
finally:
    if process.stdout is not None:
        try:
            process.stdout.close()
        except Exception:
            pass

if rc is None:
    raise RuntimeError(f"docker compose did not complete normally. See {LOG_FILE}.")
if rc != 0:
    raise RuntimeError(f"docker compose exited with {rc}. See {LOG_FILE} for full output.")
print(f"\nApp data dir: {APP_DATA_DIR}")
print("Run Section 9 to verify containers / FPS.")


## 9. Verify Deployment

Polls container status and checks FPS output from the perception container.
Re-run this cell every minute until all core containers show `Up`.

**Stack is ready when these containers show `Up`** (2D and 3D share the same container names — no `-2d` / `-3d` suffix; MV3DT containers carry a `-mv3dt` suffix):

- **Core (2D / 3D):** `vss-vios-nvstreamer`, `vss-rtvi-cv`, `vss-rtvi-cv-sdr`, `vss-configurator`, `vss-behavior-analytics`, the broker (`kafka` / `redis`) + `vss-broker-health-check`, plus the VST stack (`vss-vios-postgres`, `-sensor`, `-streamprocessing`, `-sdr`, `-mcp`, `-ingress`, `-envoy`)
- **3D adds:** `vss-rtvi-cv-config-adaptor`
- **MV3DT (`-mv3dt` suffix):** `vss-vios-nvstreamer-mv3dt`, `vss-rtvi-cv-mv3dt`, `vss-rtvi-cv-bev-fusion`, `mosquitto`, `vss-configurator-mv3dt`, `vss-behavior-analytics-mv3dt`, broker + `vss-broker-health-check`, plus the VST stack
- **`bp_wh` adds (agents/UI):** `vss-haproxy-ingress`, `vss-agent`, `vss-agent-ui`, `vss-va-mcp`, `vss-rtvi-vlm`, `vss-alert-bridge`, `phoenix`, plus the LLM NIM (container named after `LLM_NAME_SLUG`) when `LLM_MODE=local`
- **Extended profile (`MINIMAL_PROFILE=""`, kafka/redis) adds:** `vss-haproxy-ingress`, `kibana`, `logstash`, `vss-video-analytics-api` (MV3DT: `vss-video-analytics-api-mv3dt`)
- **`elasticsearch`:** deployed for `bp_wh` always, **or** kafka/redis in extended mode

> **First-run perception is slow:** `vss-rtvi-cv` compiles the TensorRT engine on its first start. The engine is cached under `$VSS_DATA_DIR/models/mtmc/` so subsequent starts skip the compile. FPS lines only appear once the engine load finishes.


In [ ]:
import subprocess, os

# ---- Container status ----
print("=== Running Containers ===")
subprocess.run(["docker", "ps", "--format", "table {{.Names}}\t{{.Status}}\t{{.Ports}}"])
print()

# ---- FPS check ----
# In 3.2 the perception container is `vss-rtvi-cv` for MODE=2d/3d and
# `vss-rtvi-cv-mv3dt` for MODE=mv3dt. mv3dt additionally runs a BEV-fusion
# container (`vss-rtvi-cv-bev-fusion`) but FPS is reported by the per-camera
# perception container.
container = "vss-rtvi-cv-mv3dt" if MODE == "mv3dt" else "vss-rtvi-cv"
print(f"=== FPS check ({container}) ===")
r = subprocess.run(
    ["docker", "logs", "--tail", "50", container],
    capture_output=True, text=True,
)
output = (r.stdout or "") + (r.stderr or "")
if r.returncode != 0:
    if output.strip():
        print(f"docker logs exit {r.returncode} for {container}:\n{output[-800:]}")
    else:
        print(f"docker logs failed for {container} (exit {r.returncode}). Is the stack up?")
else:
    # Perception logs report throughput as DeepStream PERF lines (no literal "fps").
    def _is_throughput_log_line(line: str) -> bool:
        lo = line.lower()
        return (
            "fps" in lo
            or "perf" in lo
            or "active sources" in lo
            or ("source_id" in lo and "stream_name" in lo)
        )

    perf_lines = [l for l in output.splitlines() if _is_throughput_log_line(l)]
    if perf_lines:
        for line in perf_lines[-8:]:
            print(line)
    else:
        print(f"No FPS / PERF output yet from {container}.")
        print("Re-run this cell in ~1 minute once the stack is fully up.")
        print()
        print(f"docker logs --tail 10 {container}:")
        r_tail = subprocess.run(
            ["docker", "logs", "--tail", "10", container],
            capture_output=True, text=True,
        )
        tail_out = (r_tail.stdout or "") + (r_tail.stderr or "")
        if tail_out.strip():
            print(tail_out.rstrip())
        else:
            print("(no log lines)")


## 10. Access the UI

Browser-facing services split across **one** port:

- **HAProxy ingress (`7777`)** — path-based routing for everything except VST. For the **`bp_wh` 2D agents profile**, the **Agent UI** (chat + analytics) is served at the ingress **root** `/` → `7777-<env>.brevlab.com/`. Also: **Kibana** → `/kibana/` (requires extended profile: `MINIMAL_PROFILE=""`), VST → `/VST/`. (The root `/` returns 503 on `bp_wh_kafka` / `bp_wh_redis` — no Agent UI in those profiles.)

Run the cell below to generate your URLs.

**On Brev:** create **one** secure links — for port `7777` (Kibana and the other proxied services). They resolve to `7777-<env>.brevlab.com`.

**On other cloud providers:** depending on your CSP's firewall / security-group config, expose or forward the ports below. Everything is behind `7777`.

| Port | Service | Profiles | Access |
|------|---------|----------|--------|
| 7777 | HAProxy ingress (Agent UI on `bp_wh`, Kibana, Video Analytics API, …) | all | Primary — proxied paths |
| 5601 | Kibana (direct) | extended only | Not needed (behind proxy at `/kibana/`) |
| 30888 | VST (direct) | extended only | Not needed (behind proxy at `/vst/`) |

### Streaming limitations (Live / Recorded)

VST live and recorded **video playback does not work through a Brev secure link** — the VST UI loads and stream lists/recordings are browsable, but live and recorded video will not render. This is a Brev secure-link limitation.

In [ ]:
import os

if BREV_ENV_ID:
    proxy_port = os.environ.get("PROXY_PORT", "7777")
    brev_link_prefix = os.environ.get("BREV_LINK_PREFIX", f"{proxy_port}")
    ui_url = f"https://{brev_link_prefix}-{BREV_ENV_ID}.brevlab.com"
    vst_url = f"{ui_url}/vst/"
    print("Access (via Brev secure links):")
    if BP_PROFILE == "bp_wh":
        print(f"  Agent UI:        {ui_url}/   (bp_wh 2D agents — main chat/analytics UI)")
    print(f"  Kibana:          {ui_url}/kibana/   (requires extended profile)")
    print(f"  VST UI:          {vst_url}   (direct, NOT via HAProxy)")
    print()
    print("Setup:")
    print(f"  1. Create a Brev secure link for port {proxy_port} (Kibana / other proxied services)")
    print()
    print("Note: VST live/recorded playback does NOT work through a Brev secure link.")
    print("      The UI loads and stream lists/recordings are browsable, but live and")
    print("      recorded video will not render.")
else:
    proxy_port = os.environ.get("PROXY_PORT", "7777")
    host = EXTERNAL_IP or HOST_IP
    proxy_url = f"http://{host}:{proxy_port}"
    print("Access (direct):")
    if BP_PROFILE == "bp_wh":
        print(f"  Agent UI: {proxy_url}/   (bp_wh 2D agents — main chat/analytics UI)")
    print(f"  Kibana: {proxy_url}/kibana/   (requires extended profile, via HAProxy)")
    print(f"  VST UI: {proxy_url}/vst/   (direct on port {VST_PORT}, NOT via HAProxy)")
    print()
    print("If the URLs are not directly accessible, use one of these methods:")
    print()
    print("  SSH port forwarding (works everywhere) — forward BOTH ports:")
    print(f"    ssh -L {proxy_port}:localhost:{proxy_port}")
    print(f"    Then open: http://localhost:{proxy_port}/kibana/  and  http://localhost:{proxy_port}/vst/")
    print()
    print("  VSCode Remote SSH:")
    print("    Connect to the instance via Remote-SSH; ports forward automatically.")
    print()
    print("Note: Live/recorded stream playback uses WebRTC UDP; a TCP-only tunnel of these")
    print("ports will not carry it. Open the UDP port range on your firewall.")

## 11. Next Steps

Once Kibana is accessible, open the **Discover** or **Dashboard** tab to view behavior analytics events
(ROI crossings, proximity alerts, object tracking metadata) flowing from the warehouse deployment.

For more details see the [VSS Warehouse Blueprint documentation](https://docs.nvidia.com/vss/3.2.0/).


## 12. Stop Deployment

Stops all containers **without deleting data or volumes**.
Re-run Section 8 to restart the stack.


In [ ]:
import subprocess, os

print("Stopping warehouse stack (env: industry-profiles/warehouse-operations/.env) ...")
r = subprocess.run(
    ["docker", "compose",
     "-f", "compose.yml",
     "--env-file", "industry-profiles/warehouse-operations/.env",
     "stop"],
    cwd=DEPLOYMENTS_DIR, capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("docker compose stop failed")
print("Stack stopped.")


## 13. Teardown

Stops and removes all containers, networks, and volumes. Data in `$VSS_DATA_DIR` (app data, including the cached TensorRT engine) is preserved on disk. Re-run Sections 8–9 to redeploy.


In [ ]:
import subprocess, os

ENV_REL = "industry-profiles/warehouse-operations/.env"

print("Tearing down warehouse stack ...")

for cmd in [
    ["docker", "compose", "-f", "compose.yml", "--env-file", ENV_REL, "down", "-v"],
    ["docker", "volume", "prune", "-f"],
    ["docker", "system", "prune", "-f"],
]:
    print(f"  {' '.join(cmd)}")
    r = subprocess.run(cmd, cwd=DEPLOYMENTS_DIR, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout)
    if r.returncode != 0:
        print(f"  WARNING: {r.stderr}")

# Cleanup datalog scripts — in 3.2 the script lives in-tree.
cleanup = os.path.join(DEPLOYMENTS_DIR, "scripts", "cleanup_all_datalog.sh")
if os.path.isfile(cleanup):
    print(f"\n  bash {cleanup} -e {ENV_REL}")
    r = subprocess.run(
        ["bash", cleanup, "-e", ENV_REL],
        cwd=DEPLOYMENTS_DIR, capture_output=True, text=True,
    )
    if r.stdout.strip():
        print(r.stdout)

print("Teardown complete.")


In [ ]:
# To remove the downloaded app-data tarball + extracted dir from disk:
# (the repo at DEPLOY_SOURCE_PATH is NOT removed — that's a git checkout, not a notebook artifact)
#
# import shutil
# shutil.rmtree(APP_DATA_EXTRACT_DIR)   # <home>/vss-warehouse-app-data_v<version>/
# print("App-data artifacts removed.")
